# EDA — Identifikasi Jenis Sampah, Senpro

Notebook ini berisi seluruh analisis EDA untuk dataset Senpro: distribusi kelas, deteksi duplikat (pixel-exact), properti teknis gambar, analisis kualitas, dan analisis warna/histogram HSV.

**Cara menjalankan di environment baru:** cukup ubah `ROOT_DIR` di cell konfigurasi di bawah, lalu jalankan semua cell dari atas ke bawah.

In [7]:
# ============================================================
# KONFIGURASI PATH — UBAH HANYA DI SINI
# ============================================================
# Notebook otomatis mendeteksi apakah berjalan di Google Colab
# atau di lokal. Sesuaikan ROOT_DIR di bawah sesuai environment.
#
#   Google Colab : ganti ke path Google Drive kamu, misal
#                  /content/drive/MyDrive/Senpro
#   Lokal        : biarkan sebagai str(Path(".").resolve().parent)
#                  supaya otomatis naik satu level dari EDA/ ke folder
#                  root Senpro/ (tempat folder Dataset/ berada)
# ============================================================

import sys
import os
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT_DIR = "/content/drive/MyDrive/Senpro"  # <-- Ganti path Drive kamu di sini
else:
    # Di lokal: naik satu level dari folder EDA/ ke folder root Senpro/
    ROOT_DIR = str(Path(".").resolve().parent)

# dataset_dir dipakai oleh semua cell di bawah, tidak perlu diubah lagi
dataset_dir = Path(ROOT_DIR) / "Dataset"

print(f"ROOT_DIR   : {ROOT_DIR}")
print(f"dataset_dir: {dataset_dir}")
print(f"dataset_dir ada: {dataset_dir.exists()}")

ROOT_DIR   : D:\Bagas\Satria_Data_BDC\BDC 2026\Senpro
dataset_dir: D:\Bagas\Satria_Data_BDC\BDC 2026\Senpro\Dataset
dataset_dir ada: True


## 1. Distribusi Kelas

Menghitung jumlah gambar per kelas dan memeriksa apakah terdapat *class imbalance* yang signifikan.

In [8]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import os

# dataset_dir sudah didefinisikan di cell konfigurasi di atas

# Ambil semua subfolder kelas (kecuali Archive)
EXCLUDE_FOLDERS = {"Archive"}
class_dirs = sorted([
    p for p in dataset_dir.iterdir()
    if p.is_dir() and p.name not in EXCLUDE_FOLDERS
])

def count_images(folder):
    return sum(1 for p in folder.rglob("*") if p.is_file())

# Buat dataframe jumlah data per label
data = []
for folder in class_dirs:
    data.append({
        "label": folder.name,
        "count": count_images(folder)
    })

df = pd.DataFrame(data)

# Bersihkan nama label agar lebih rapi (hapus prefix angka_)
df["class_name"] = df["label"].str.split("_", n=1).str[1].fillna(df["label"])
df["percentage"] = (df["count"] / df["count"].sum() * 100).round(2)

print("Jumlah data per label:")
print(df[["class_name", "count", "percentage"]])

# Cek tingkat imbalance
min_count = df["count"].min()
max_count = df["count"].max()
imbalance_ratio = max_count / min_count if min_count > 0 else float("inf")

print("\nRingkasan:")
print(f"Min count: {min_count}")
print(f"Max count: {max_count}")
print(f"Imbalance ratio (max/min): {imbalance_ratio:.2f}")

if imbalance_ratio < 1.2:
    print("Hasil: dataset cenderung seimbang.")
elif imbalance_ratio < 2:
    print("Hasil: dataset sedikit tidak seimbang.")
else:
    print("Hasil: dataset mengalami class imbalance yang signifikan.")

# Visualisasi distribusi
colors = ["#4C78A8", "#54A24B", "#E45756", "#F58518", "#B279A2"]
plt.figure(figsize=(8, 4))
plt.bar(df["class_name"], df["count"], color=colors[:len(df)])
plt.title("Distribusi jumlah data per label")
plt.xlabel("Label")
plt.ylabel("Jumlah data")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

Jumlah data per label:
  class_name  count  percentage
0  Recylable  12973       42.07
1    Organik  13800       44.75
2         B3   4067       13.19

Ringkasan:
Min count: 4067
Max count: 13800
Imbalance ratio (max/min): 3.39
Hasil: dataset mengalami class imbalance yang signifikan.


C:\Users\bagas\AppData\Local\Temp\ipykernel_26080\2376280507.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Deteksi Duplikat Gambar (Pixel-Exact)

Mencari gambar yang benar-benar identik berdasarkan nilai piksel menggunakan SHA-256 hash dari data gambar yang sudah di-load.

In [9]:
from pathlib import Path
import hashlib
from PIL import Image

# dataset_dir sudah didefinisikan di cell konfigurasi di atas
EXCLUDE_FOLDERS = {"Archive"}
class_dirs = sorted([
    p for p in dataset_dir.iterdir()
    if p.is_dir() and p.name not in EXCLUDE_FOLDERS
])
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def pixel_signature(path):
    img = Image.open(path).convert("RGB")
    width, height = img.size
    return hashlib.sha256(f"{width}x{height}".encode() + b"|" + img.tobytes()).hexdigest()

for class_dir in class_dirs:
    image_paths = sorted([
        p for p in class_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ])

    signature_map = {}
    for p in image_paths:
        try:
            sig = pixel_signature(p)
            signature_map.setdefault(sig, []).append(p)
        except Exception as e:
            print(f"Lewati {p.name}: {e}")

    duplicate_groups = {
        sig: files for sig, files in signature_map.items() if len(files) > 1
    }

    print(f"\nKelas: {class_dir.name}")
    print(f"Total gambar: {len(image_paths)}")
    print(f"Jumlah unik berdasarkan piksel: {len(signature_map)}")

    if duplicate_groups:
        print("Ditemukan gambar yang sama persis:")
        for sig, files in duplicate_groups.items():
            print(" -", [f.name for f in files])
    else:
        print("Tidak ada gambar yang sama persis berdasarkan nilai piksel.")


Kelas: 0_Recylable
Total gambar: 12973
Jumlah unik berdasarkan piksel: 12944
Ditemukan gambar yang sama persis:
 - ['clothes2051.jpg', 'clothes5234.jpg']
 - ['disadvantages-of-reusing-cooking-oil-Southern-Green_jpeg.rf.176910eb2bbb98a36b3209c5109eff60.jpg', 'disadvantages-of-reusing-cooking-oil-Southern-Green_jpeg.rf.4fe95dc404dd084e2d5cf172f3caaa97.jpg']
 - ['Image_104.png', 'Image_204.png']
 - ['Image_109.png', 'Image_205.png']
 - ['Image_110.png', 'Image_77.png']
 - ['Image_114.png', 'Image_128.png']
 - ['Image_117.png', 'Image_225.png']
 - ['Image_121.png', 'Image_145.png']
 - ['Image_132.png', 'Image_201.png']
 - ['Image_136.png', 'Image_218.png']
 - ['Image_152.png', 'Image_22.png']
 - ['Image_171.png', 'Image_187.png']
 - ['Image_244.png', 'Image_94.png']
 - ['Image_6.png', 'Image_67.png']
 - ['images-13-_jpg.rf.3dc903ffb4b8e2d109dfe38b4a3fe150.jpg', 'images-13-_jpg.rf.900ba688e8f138bc106f15268726fa13.jpg']
 - ['images-4-_jpg.rf.e8c30f4a0880595e3b14163f27427e0b.jpg', 'images-4-

## 3. Properti Teknis Gambar

Menganalisis properti teknis tiap gambar: resolusi (lebar × tinggi), rasio aspek, format file (jpg/png/dll), dan mode warna (RGB/RGBA/dll). Hasil detail disimpan ke CSV.

In [10]:
"""
EDA Properti Teknis Gambar — Senpro

Yang dianalisis:
1. Ukuran resolusi tiap citra (lebar dan tinggi)
2. Rasio aspek gambar
3. Format file (jpg, png, dll)
4. Mode warna (RGB, grayscale, RGBA, dll)

Hasil akhir berupa tabel ringkasan dan file CSV detail per gambar.
"""

import os
from pathlib import Path
from collections import Counter
import pandas as pd
from PIL import Image

# ==== KONFIGURASI ====
# dataset_dir sudah didefinisikan di cell konfigurasi di atas
OUTPUT_CSV = "eda_properti_teknis.csv"
EXCLUDE_FOLDERS = {"Archive"}

# Mapping folder → label (disesuaikan dengan struktur Dataset Senpro)
CLASS_FOLDERS = {
    folder.name: folder.name.split("_", 1)[1] if "_" in folder.name else folder.name
    for folder in sorted(dataset_dir.iterdir())
    if folder.is_dir() and folder.name not in EXCLUDE_FOLDERS
}
print("CLASS_FOLDERS:", CLASS_FOLDERS)


def kumpulkan_data_properti(dataset_dir):
    """
    Membaca seluruh gambar pada tiap subfolder kelas,
    lalu mengumpulkan properti teknisnya.
    """
    hasil = []

    for folder_name, label in CLASS_FOLDERS.items():
        folder_path = Path(dataset_dir) / folder_name
        if not folder_path.exists():
            print(f"Peringatan: folder {folder_path} tidak ditemukan, dilewati.")
            continue

        file_list = [
            f for f in os.listdir(folder_path)
            if os.path.isfile(folder_path / f)
        ]

        for file_name in file_list:
            file_path = folder_path / file_name
            ekstensi = file_name.split(".")[-1].lower() if "." in file_name else "tanpa_ekstensi"

            try:
                with Image.open(file_path) as img:
                    lebar, tinggi = img.size
                    mode_warna = img.mode
                    rasio_aspek = round(lebar / tinggi, 3) if tinggi != 0 else None

                hasil.append({
                    "nama_file": file_name,
                    "label": label,
                    "lebar": lebar,
                    "tinggi": tinggi,
                    "rasio_aspek": rasio_aspek,
                    "format_file": ekstensi,
                    "mode_warna": mode_warna,
                    "status": "ok",
                })

            except Exception as e:
                hasil.append({
                    "nama_file": file_name,
                    "label": label,
                    "lebar": None,
                    "tinggi": None,
                    "rasio_aspek": None,
                    "format_file": ekstensi,
                    "mode_warna": None,
                    "status": f"gagal_dibaca: {e}",
                })

    return pd.DataFrame(hasil)


def ringkasan_per_kelas(df):
    """
    Menampilkan ringkasan statistik resolusi dan rasio aspek per kelas.
    """
    df_valid = df[df["status"] == "ok"]

    print("\n=== RINGKASAN RESOLUSI PER KELAS ===")
    ringkasan = df_valid.groupby("label").agg(
        jumlah_gambar=("nama_file", "count"),
        lebar_min=("lebar", "min"),
        lebar_max=("lebar", "max"),
        lebar_rata=("lebar", "mean"),
        tinggi_min=("tinggi", "min"),
        tinggi_max=("tinggi", "max"),
        tinggi_rata=("tinggi", "mean"),
        rasio_aspek_min=("rasio_aspek", "min"),
        rasio_aspek_max=("rasio_aspek", "max"),
        rasio_aspek_rata=("rasio_aspek", "mean"),
    ).round(2)

    print(ringkasan)
    return ringkasan


def ringkasan_format_dan_mode(df):
    """
    Menampilkan distribusi format file dan mode warna, keseluruhan dan per kelas.
    """
    print("\n=== DISTRIBUSI FORMAT FILE (KESELURUHAN) ===")
    print(df["format_file"].value_counts())

    print("\n=== DISTRIBUSI MODE WARNA (KESELURUHAN) ===")
    print(df["mode_warna"].value_counts())

    print("\n=== DISTRIBUSI MODE WARNA PER KELAS ===")
    print(df.groupby(["label", "mode_warna"]).size())

    print("\n=== DISTRIBUSI FORMAT FILE PER KELAS ===")
    print(df.groupby(["label", "format_file"]).size())


def cek_gambar_gagal_dibaca(df):
    """
    Menampilkan daftar file yang gagal dibaca atau corrupt.
    """
    df_gagal = df[df["status"] != "ok"]
    print(f"\n=== JUMLAH GAMBAR GAGAL DIBACA: {len(df_gagal)} ===")
    if len(df_gagal) > 0:
        print(df_gagal[["nama_file", "label", "status"]])
    return df_gagal


def cek_variasi_resolusi(df):
    """
    Mengecek seberapa banyak variasi ukuran resolusi unik yang ada,
    untuk membantu menentukan ukuran resize yang tepat.
    """
    df_valid = df[df["status"] == "ok"]
    kombinasi_ukuran = df_valid.groupby(["lebar", "tinggi"]).size().sort_values(ascending=False)

    print(f"\n=== JUMLAH KOMBINASI UKURAN UNIK: {kombinasi_ukuran.shape[0]} ===")
    print("10 ukuran paling sering muncul:")
    print(kombinasi_ukuran.head(10))


def main():
    df = kumpulkan_data_properti(dataset_dir)

    if df.empty:
        print("Tidak ada data yang berhasil dikumpulkan. Cek kembali path dataset_dir.")
        return

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Detail properti tiap gambar disimpan pada {OUTPUT_CSV}")

    ringkasan_per_kelas(df)
    ringkasan_format_dan_mode(df)
    cek_variasi_resolusi(df)
    cek_gambar_gagal_dibaca(df)


if __name__ == "__main__":
    main()

CLASS_FOLDERS: {'0_Recylable': 'Recylable', '1_Organik': 'Organik', '2_B3': 'B3'}
Detail properti tiap gambar disimpan pada eda_properti_teknis.csv

=== RINGKASAN RESOLUSI PER KELAS ===
           jumlah_gambar  lebar_min  lebar_max  lebar_rata  tinggi_min  \
label                                                                    
B3                  4067         11       4032      360.71           9   
Organik            13800         51        600      260.37          69   
Recylable          12973         56        888      280.50          88   

           tinggi_max  tinggi_rata  rasio_aspek_min  rasio_aspek_max  \
label                                                                  
B3               3024       301.10             0.13             6.24   
Organik           383       198.32             0.13             8.70   
Recylable         936       275.24             0.16             6.47   

           rasio_aspek_rata  
label                        
B3                    

## 4. Analisis Kualitas Gambar

Memeriksa kualitas tiap gambar meliputi:
1. Deteksi gambar corrupt atau gagal dibuka.
2. Deteksi blur menggunakan variance dari operator Laplacian (nilai rendah → gambar blur).
3. Deteksi gambar terlalu gelap atau terlalu terang berdasarkan rata-rata intensitas piksel.
4. Deteksi duplikat identik menggunakan hash MD5 dari isi file.

In [11]:
"""
EDA Analisis Kualitas Gambar — Senpro

Script ini memeriksa kualitas tiap gambar pada dataset, meliputi
lima aspek berikut.

Aspek pertama adalah deteksi gambar corrupt atau gagal dibuka.
Aspek kedua adalah deteksi blur menggunakan variance dari operator Laplacian.
Semakin rendah nilainya, semakin besar kemungkinan gambar tersebut blur.
Aspek ketiga adalah deteksi gambar terlalu gelap atau terlalu terang
berdasarkan rata rata intensitas piksel.
Aspek keempat adalah deteksi duplikat identik menggunakan hash MD5 dari isi
file, cocok untuk menemukan file yang benar benar sama persis.

dataset_dir menggunakan variabel yang sama seperti pada cell konfigurasi.
"""

import os
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image

DATASET_DIR = dataset_dir
OUTPUT_KUALITAS_CSV = "eda_kualitas_gambar.csv"
OUTPUT_DUPLIKAT_IDENTIK_CSV = "eda_duplikat_identik.csv"

EXCLUDE_FOLDERS = {"Archive"}
CLASS_FOLDERS = {
    folder.name: folder.name.split("_", 1)[1] if "_" in folder.name else folder.name
    for folder in sorted(dataset_dir.iterdir())
    if folder.is_dir() and folder.name not in EXCLUDE_FOLDERS
}

AMBANG_BLUR = 100.0
AMBANG_GELAP = 40.0
AMBANG_TERANG = 220.0


def hitung_md5_file(path_file):
    with open(path_file, "rb") as f:
        konten = f.read()
    return hashlib.md5(konten).hexdigest()


def periksa_satu_gambar(path_gambar):
    try:
        with Image.open(path_gambar) as img_pil:
            img_pil.verify()

        with Image.open(path_gambar) as img_pil:
            img_pil_rgb = img_pil.convert("RGB")
            array_rgb = np.array(img_pil_rgb)
            img_gray = cv2.cvtColor(array_rgb, cv2.COLOR_RGB2GRAY)

            skor_blur = cv2.Laplacian(img_gray, cv2.CV_64F).var()
            rata_kecerahan = float(img_gray.mean())

            md5_hash = hitung_md5_file(path_gambar)

            status_blur = skor_blur < AMBANG_BLUR
            status_gelap = rata_kecerahan < AMBANG_GELAP
            status_terang = rata_kecerahan > AMBANG_TERANG

            return {
                "status": "ok",
                "skor_blur": round(float(skor_blur), 2),
                "rata_kecerahan": round(rata_kecerahan, 2),
                "terindikasi_blur": bool(status_blur),
                "terindikasi_gelap": bool(status_gelap),
                "terindikasi_terang": bool(status_terang),
                "md5_hash": md5_hash,
            }

    except Exception as e:
        return {
            "status": f"corrupt_atau_gagal_dibaca: {e}",
            "skor_blur": None,
            "rata_kecerahan": None,
            "terindikasi_blur": None,
            "terindikasi_gelap": None,
            "terindikasi_terang": None,
            "md5_hash": None,
        }


def proses_seluruh_dataset(dataset_dir):
    hasil = []

    for folder_name, label in CLASS_FOLDERS.items():
        folder_path = Path(dataset_dir) / folder_name

        if not folder_path.exists():
            print(f"Peringatan, folder {folder_path} tidak ditemukan, dilewati")
            continue

        file_list = [
            f for f in os.listdir(folder_path)
            if os.path.isfile(folder_path / f)
        ]

        print(f"Memproses kelas {label}, jumlah gambar {len(file_list)}")

        for idx, file_name in enumerate(file_list, start=1):
            file_path = folder_path / file_name
            hasil_periksa = periksa_satu_gambar(file_path)
            hasil_periksa["nama_file"] = file_name
            hasil_periksa["label"] = label
            hasil.append(hasil_periksa)

            if idx % 500 == 0 or idx == len(file_list):
                print(f"  kelas {label}, gambar ke {idx} dari {len(file_list)} selesai diperiksa")

    return pd.DataFrame(hasil)


def ringkasan_kualitas(df):
    df_valid = df[df["status"] == "ok"]
    df_corrupt = df[df["status"] != "ok"]

    print(f"\nJumlah gambar corrupt atau gagal dibaca, total {len(df_corrupt)}")
    if len(df_corrupt) > 0:
        print(df_corrupt[["nama_file", "label", "status"]])

    print("\nRingkasan jumlah gambar terindikasi blur per kelas")
    print(df_valid.groupby("label")["terindikasi_blur"].sum())

    print("\nRingkasan jumlah gambar terindikasi terlalu gelap per kelas")
    print(df_valid.groupby("label")["terindikasi_gelap"].sum())

    print("\nRingkasan jumlah gambar terindikasi terlalu terang per kelas")
    print(df_valid.groupby("label")["terindikasi_terang"].sum())

    print("\nStatistik skor blur per kelas")
    print(df_valid.groupby("label")["skor_blur"].describe().round(2))

    print("\nStatistik rata rata kecerahan per kelas")
    print(df_valid.groupby("label")["rata_kecerahan"].describe().round(2))


def deteksi_duplikat_identik(df):
    """
    Mencari gambar dengan hash MD5 sama persis, menandakan file identik
    baik dalam kelas yang sama maupun berbeda kelas.
    """
    df_valid = df[df["status"] == "ok"]

    duplikat_grup = df_valid.groupby("md5_hash").filter(lambda x: len(x) > 1)
    hasil = duplikat_grup.sort_values("md5_hash")[["nama_file", "label", "md5_hash"]]

    print(f"\nJumlah gambar yang termasuk grup duplikat identik, total {len(hasil)}")
    if len(hasil) > 0:
        print(hasil)

    return hasil


def main():
    df = proses_seluruh_dataset(DATASET_DIR)

    if df.empty:
        print("Tidak ada data yang berhasil diproses. Cek kembali path dataset_dir")
        return

    df.to_csv(OUTPUT_KUALITAS_CSV, index=False)
    print(f"Detail kualitas tiap gambar disimpan pada {OUTPUT_KUALITAS_CSV}")

    ringkasan_kualitas(df)

    df_duplikat_identik = deteksi_duplikat_identik(df)
    df_duplikat_identik.to_csv(OUTPUT_DUPLIKAT_IDENTIK_CSV, index=False)
    print(f"Daftar duplikat identik disimpan pada {OUTPUT_DUPLIKAT_IDENTIK_CSV}")


if __name__ == "__main__":
    main()

Memproses kelas Recylable, jumlah gambar 12973
  kelas Recylable, gambar ke 500 dari 12973 selesai diperiksa
  kelas Recylable, gambar ke 1000 dari 12973 selesai diperiksa
  kelas Recylable, gambar ke 1500 dari 12973 selesai diperiksa
  kelas Recylable, gambar ke 2000 dari 12973 selesai diperiksa
  kelas Recylable, gambar ke 2500 dari 12973 selesai diperiksa
  kelas Recylable, gambar ke 3000 dari 12973 selesai diperiksa
  kelas Recylable, gambar ke 3500 dari 12973 selesai diperiksa
  kelas Recylable, gambar ke 4000 dari 12973 selesai diperiksa
  kelas Recylable, gambar ke 4500 dari 12973 selesai diperiksa
  kelas Recylable, gambar ke 5000 dari 12973 selesai diperiksa
  kelas Recylable, gambar ke 5500 dari 12973 selesai diperiksa
  kelas Recylable, gambar ke 6000 dari 12973 selesai diperiksa
  kelas Recylable, gambar ke 6500 dari 12973 selesai diperiksa
  kelas Recylable, gambar ke 7000 dari 12973 selesai diperiksa
  kelas Recylable, gambar ke 7500 dari 12973 selesai diperiksa
  kelas R

## 5. Analisis Warna — Histogram HSV Antar Kelas

Menghitung histogram warna tiap gambar pada ruang HSV, lalu menggabungkan histogram per kelas untuk melihat pola warna dominan masing-masing kelas. Overlap antar histogram kelas diukur menggunakan *histogram intersection* dan jarak *Bhattacharyya*:
- Intersection mendekati 1 → histogram sangat mirip.
- Jarak Bhattacharyya mendekati 0 → sangat mirip.

In [12]:
"""
EDA Analisis Warna dan Tekstur, Histogram Warna Antar Kelas — Senpro

Script ini menghitung histogram warna tiap gambar pada ruang HSV, lalu
menggabungkan histogram tersebut per kelas untuk melihat pola warna
dominan masing masing kelas. Setelah itu dihitung seberapa besar overlap
antar histogram kelas menggunakan histogram intersection dan jarak
Bhattacharyya, sehingga bisa dipastikan apakah warna antar kelas memang
berbeda jelas atau justru saling tumpang tindih.

dataset_dir menggunakan variabel yang sama seperti pada cell konfigurasi.
"""

import os
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

DATASET_DIR = dataset_dir
OUTPUT_HISTOGRAM_CSV = "eda_histogram_warna_per_kelas.csv"
OUTPUT_OVERLAP_CSV = "eda_overlap_warna_antar_kelas.csv"
OUTPUT_PLOT_HUE = "eda_histogram_hue_per_kelas.png"
OUTPUT_PLOT_SATURATION = "eda_histogram_saturation_per_kelas.png"
OUTPUT_PLOT_VALUE = "eda_histogram_value_per_kelas.png"

EXCLUDE_FOLDERS = {"Archive"}
CLASS_FOLDERS = {
    folder.name: folder.name.split("_", 1)[1] if "_" in folder.name else folder.name
    for folder in sorted(dataset_dir.iterdir())
    if folder.is_dir() and folder.name not in EXCLUDE_FOLDERS
}

UKURAN_RESIZE = 128
BINS_HUE = 30
BINS_SATURATION = 32
BINS_VALUE = 32


def muat_gambar_hsv(path_gambar):
    img = cv2.imread(str(path_gambar))
    if img is None:
        raise ValueError("gambar gagal dibaca")
    img = cv2.resize(img, (UKURAN_RESIZE, UKURAN_RESIZE))
    img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    return img_hsv


def hitung_histogram_satu_gambar(img_hsv):
    hist_hue = cv2.calcHist([img_hsv], [0], None, [BINS_HUE], [0, 180])
    hist_saturation = cv2.calcHist([img_hsv], [1], None, [BINS_SATURATION], [0, 256])
    hist_value = cv2.calcHist([img_hsv], [2], None, [BINS_VALUE], [0, 256])

    hist_hue = cv2.normalize(hist_hue, hist_hue).flatten()
    hist_saturation = cv2.normalize(hist_saturation, hist_saturation).flatten()
    hist_value = cv2.normalize(hist_value, hist_value).flatten()

    return hist_hue, hist_saturation, hist_value


def kumpulkan_histogram_seluruh_dataset(dataset_dir):
    akumulasi_hue = {label: np.zeros(BINS_HUE) for label in CLASS_FOLDERS.values()}
    akumulasi_saturation = {label: np.zeros(BINS_SATURATION) for label in CLASS_FOLDERS.values()}
    akumulasi_value = {label: np.zeros(BINS_VALUE) for label in CLASS_FOLDERS.values()}
    jumlah_gambar_per_kelas = {label: 0 for label in CLASS_FOLDERS.values()}

    for folder_name, label in CLASS_FOLDERS.items():
        folder_path = Path(dataset_dir) / folder_name

        if not folder_path.exists():
            print(f"Peringatan, folder {folder_path} tidak ditemukan, dilewati")
            continue

        file_list = [
            f for f in os.listdir(folder_path)
            if os.path.isfile(folder_path / f)
        ]

        print(f"Memproses kelas {label}, jumlah gambar {len(file_list)}")

        for idx, file_name in enumerate(file_list, start=1):
            file_path = folder_path / file_name

            try:
                img_hsv = muat_gambar_hsv(file_path)
                hist_hue, hist_saturation, hist_value = hitung_histogram_satu_gambar(img_hsv)

                akumulasi_hue[label] += hist_hue
                akumulasi_saturation[label] += hist_saturation
                akumulasi_value[label] += hist_value
                jumlah_gambar_per_kelas[label] += 1

            except Exception as e:
                print(f"Gagal memproses {file_path}: {e}")

            if idx % 1000 == 0 or idx == len(file_list):
                print(f"  kelas {label}, gambar ke {idx} dari {len(file_list)} selesai diproses")

    hasil_rata_rata = {}
    for label in CLASS_FOLDERS.values():
        jumlah = jumlah_gambar_per_kelas[label]
        if jumlah == 0:
            continue

        hasil_rata_rata[label] = {
            "hue": akumulasi_hue[label] / jumlah,
            "saturation": akumulasi_saturation[label] / jumlah,
            "value": akumulasi_value[label] / jumlah,
            "jumlah_gambar": jumlah,
        }

    return hasil_rata_rata


def simpan_histogram_ke_csv(hasil_rata_rata, output_path):
    daftar_baris = []

    for label, data in hasil_rata_rata.items():
        for idx_bin, nilai in enumerate(data["hue"]):
            daftar_baris.append({"label": label, "channel": "hue", "bin": idx_bin, "nilai": float(nilai)})
        for idx_bin, nilai in enumerate(data["saturation"]):
            daftar_baris.append({"label": label, "channel": "saturation", "bin": idx_bin, "nilai": float(nilai)})
        for idx_bin, nilai in enumerate(data["value"]):
            daftar_baris.append({"label": label, "channel": "value", "bin": idx_bin, "nilai": float(nilai)})

    df = pd.DataFrame(daftar_baris)
    df.to_csv(output_path, index=False)
    print(f"Data histogram per kelas disimpan pada {output_path}")

    return df


def hitung_overlap_antar_kelas(hasil_rata_rata):
    """
    Menghitung histogram intersection dan jarak Bhattacharyya antar
    pasangan kelas, untuk tiap channel warna hue, saturation, dan value.
    Nilai intersection mendekati satu berarti dua histogram sangat mirip.
    Nilai jarak Bhattacharyya mendekati nol juga berarti sangat mirip.
    """
    daftar_kelas = list(hasil_rata_rata.keys())
    hasil = []

    for kelas_a, kelas_b in combinations(daftar_kelas, 2):
        for channel in ["hue", "saturation", "value"]:
            hist_a = hasil_rata_rata[kelas_a][channel].astype(np.float32)
            hist_b = hasil_rata_rata[kelas_b][channel].astype(np.float32)

            intersection = cv2.compareHist(hist_a, hist_b, cv2.HISTCMP_INTERSECT)
            bhattacharyya = cv2.compareHist(hist_a, hist_b, cv2.HISTCMP_BHATTACHARYYA)
            correlation = cv2.compareHist(hist_a, hist_b, cv2.HISTCMP_CORREL)

            hasil.append({
                "kelas_a": kelas_a,
                "kelas_b": kelas_b,
                "channel": channel,
                "histogram_intersection": round(float(intersection), 4),
                "jarak_bhattacharyya": round(float(bhattacharyya), 4),
                "korelasi": round(float(correlation), 4),
                "overlap_tinggi": bool(bhattacharyya < 0.3),
            })

    df_overlap = pd.DataFrame(hasil)
    return df_overlap


def buat_plot_histogram(hasil_rata_rata, channel, jumlah_bin, rentang_maks, output_path, label_sumbu_x):
    plt.figure(figsize=(8, 5))

    for label, data in hasil_rata_rata.items():
        sumbu_x = np.linspace(0, rentang_maks, jumlah_bin)
        plt.plot(sumbu_x, data[channel], label=label)

    plt.title(f"Rata Rata Histogram {channel.capitalize()} per Kelas")
    plt.xlabel(label_sumbu_x)
    plt.ylabel("Frekuensi Ternormalisasi")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()

    print(f"Plot histogram {channel} disimpan pada {output_path}")


def main():
    hasil_rata_rata = kumpulkan_histogram_seluruh_dataset(DATASET_DIR)

    if len(hasil_rata_rata) == 0:
        print("Tidak ada data yang berhasil diproses. Cek kembali path dataset_dir")
        return

    simpan_histogram_ke_csv(hasil_rata_rata, OUTPUT_HISTOGRAM_CSV)

    df_overlap = hitung_overlap_antar_kelas(hasil_rata_rata)
    df_overlap.to_csv(OUTPUT_OVERLAP_CSV, index=False)
    print(f"\nHasil overlap warna antar kelas disimpan pada {OUTPUT_OVERLAP_CSV}")
    print(df_overlap)

    print("\nRingkasan pasangan kelas dan channel dengan overlap tinggi (jarak Bhattacharyya di bawah 0.3)")
    print(df_overlap[df_overlap["overlap_tinggi"]])

    buat_plot_histogram(hasil_rata_rata, "hue", BINS_HUE, 180, OUTPUT_PLOT_HUE, "Nilai Hue")
    buat_plot_histogram(hasil_rata_rata, "saturation", BINS_SATURATION, 256, OUTPUT_PLOT_SATURATION, "Nilai Saturation")
    buat_plot_histogram(hasil_rata_rata, "value", BINS_VALUE, 256, OUTPUT_PLOT_VALUE, "Nilai Value")


if __name__ == "__main__":
    main()

Memproses kelas Recylable, jumlah gambar 12973
  kelas Recylable, gambar ke 1000 dari 12973 selesai diproses
  kelas Recylable, gambar ke 2000 dari 12973 selesai diproses
  kelas Recylable, gambar ke 3000 dari 12973 selesai diproses
  kelas Recylable, gambar ke 4000 dari 12973 selesai diproses
  kelas Recylable, gambar ke 5000 dari 12973 selesai diproses
  kelas Recylable, gambar ke 6000 dari 12973 selesai diproses
  kelas Recylable, gambar ke 7000 dari 12973 selesai diproses
  kelas Recylable, gambar ke 8000 dari 12973 selesai diproses
  kelas Recylable, gambar ke 9000 dari 12973 selesai diproses
  kelas Recylable, gambar ke 10000 dari 12973 selesai diproses
  kelas Recylable, gambar ke 11000 dari 12973 selesai diproses
  kelas Recylable, gambar ke 12000 dari 12973 selesai diproses
  kelas Recylable, gambar ke 12973 dari 12973 selesai diproses
Memproses kelas Organik, jumlah gambar 13800
  kelas Organik, gambar ke 1000 dari 13800 selesai diproses
  kelas Organik, gambar ke 2000 dari 1